In [21]:
# Clear any cached module imports
import sys
# Remove any previously imported strategy modules from cache
modules_to_remove = [k for k in sys.modules.keys() if 'supply_demand' in k]
for mod in modules_to_remove:
    del sys.modules[mod]
print(f"✓ Cleared {len(modules_to_remove)} cached modules")

✓ Cleared 3 cached modules


# Supply & Demand V1 Strategy Backtest

This notebook demonstrates the Supply and Demand zone-based trading strategy.

**Strategy Overview:**
- Identifies supply/demand zones using Drop-Base-Rally (DBR) and Rally-Base-Drop (RBD) patterns
- Scores zones using odds enhancers (freshness, leg-out strength, base length)
- Enters trades at proximal lines with stops at distal lines
- Targets minimum 3R (3x risk-to-reward ratio)

## Key Parameters (Edit in first cell)
All configurable parameters are in the "Parameters Configuration" cell below.

**Note:** If you encounter import errors, run the first cell to clear cached modules, then restart the kernel.

## Parameters Configuration

**Edit these parameters to customize the backtest:**

In [22]:
# ============================================================================
# BACKTEST CONFIGURATION - Edit these parameters
# ============================================================================

# Trading pairs to test
SYMBOLS = ['BTC/USDT', 'ETH/USDT']

# Date range for backtest (YYYY-MM-DD)
START_DATE = '2024-01-01'
END_DATE = '2024-12-31'

# Initial account size
INITIAL_CAPITAL = 10000.0  # USD

# Strategy parameters (from SupplyDemandParameters)
STRATEGY_PARAMS = {
    # Candle classification
    'boring_body_ratio': 0.50,      # body <= 50% of range = boring
    'exciting_body_ratio': 0.50,    # body > 50% of range = exciting
    
    # Zone detection
    'min_base_candles': 1,
    'max_base_candles': 6,
    'min_legout_candles': 1,
    
    # Proximal line placement
    'proximal_mode': 'body',        # 'body' or 'wick'
    
    # Scoring thresholds
    'min_setup_score': 6.0,         # Minimum score to take trade
    'freshness_touches_best': 0,    # Fresh = 3 points
    'freshness_touches_good': 1,    # 1 touch = 1.5 points
    'base_time_best': 3,            # ≤3 candles = 2 points
    'base_time_good': 6,            # 4-6 candles = 1 point
    'legout_strength_high_threshold': 0.10,  # 10% = 2 points
    'legout_strength_mid_threshold': 0.05,   # 5% = 1 point
    
    # Trade management
    'risk_pct': 0.02,               # 2% risk per trade
    'breakeven_at_r': 2.0,          # Move stop to BE at 2R
    'take_profit_at_r': 3.0,        # Take profit at 3R
    'min_reward_risk': 3.0,         # Minimum 3:1 R:R
    'stop_buffer_pct': 0.001,       # 0.1% buffer on stop
    
    # Multi-timeframe (for this demo, using single timeframe)
    'htf_tf': '4h',
    'itf_tf': '1h',
    'ltf_tf': '15m',
    'rtf_tf': '5m',
    
    # Trend detection
    'pivot_len': 5,
    'pivots_to_consider': 4,
}

print("✓ Parameters configured")
print(f"  Symbols: {SYMBOLS}")
print(f"  Date range: {START_DATE} to {END_DATE}")
print(f"  Initial capital: ${INITIAL_CAPITAL:,.2f}")

✓ Parameters configured
  Symbols: ['BTC/USDT', 'ETH/USDT']
  Date range: 2024-01-01 to 2024-12-31
  Initial capital: $10,000.00


## Setup and Imports

In [23]:
# Restart kernel to clear cached imports
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Add repository root to path so packaged modules resolve in notebooks and terminals
repo_root = os.path.abspath("../..")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import strategy module from the packaged path
from strategies.supply_demand_v1.strategy import (
    SupplyDemandParameters,
    detect_zones_dbr_rbd,
    find_nearest_fresh_zones_htf,
    classify_curve,
    classify_trend,
    should_allow_trade,
    detect_pivot_highs_lows,
    curve_location,
    trend_direction_itf,
    is_zone_fresh,
    odds_enhancer_score,
    build_trade_plan,
    calculate_r_multiple,
    Zone,
    ZoneType,
    CurveLocation,
    TrendDirection,
)


# Import integrity validation module
from strategies.supply_demand_v1.integrity import (
    run_integrity_checks,
    print_integrity_report,
)

print("✓ Imports successful")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Imports successful


## Generate Synthetic Market Data

For this demo, we'll generate synthetic candle data that includes supply/demand patterns.
In a real backtest, you would load actual market data from an exchange or data provider.

In [24]:
def generate_synthetic_candles(symbol: str, num_candles: int = 200, base_price: float = 50000) -> List[Dict]:
    """Generate synthetic OHLC candles with embedded supply/demand patterns"""
    np.random.seed(42 if symbol == 'BTC/USDT' else 43)
    candles = []
    current_price = base_price
    
    for i in range(num_candles):
        # Determine if we should create a pattern
        if i % 30 == 0 and i > 0:
            # Create DBR pattern (demand zone)
            # Exciting drop
            open_price = current_price
            close_price = current_price * 0.95
            high_price = open_price
            low_price = close_price * 0.99
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i)
            })
            current_price = close_price
            
            # Boring base (2-3 candles)
            for j in range(2):
                open_price = current_price
                close_price = current_price + np.random.uniform(-20, 20)
                high_price = max(open_price, close_price) * 1.002
                low_price = min(open_price, close_price) * 0.998
                candles.append({
                    'open': open_price, 'high': high_price,
                    'low': low_price, 'close': close_price,
                    'timestamp': datetime(2024, 1, 1) + timedelta(hours=i+j+1)
                })
                current_price = close_price
            
            # Exciting rally
            open_price = current_price
            close_price = current_price * 1.08
            high_price = close_price
            low_price = open_price
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i+3)
            })
            current_price = close_price
            
        elif i % 25 == 15 and i > 15:
            # Create RBD pattern (supply zone)
            # Exciting rally
            open_price = current_price
            close_price = current_price * 1.05
            high_price = close_price * 1.01
            low_price = open_price
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i)
            })
            current_price = close_price
            
            # Boring base
            for j in range(2):
                open_price = current_price
                close_price = current_price + np.random.uniform(-30, 30)
                high_price = max(open_price, close_price) * 1.003
                low_price = min(open_price, close_price) * 0.997
                candles.append({
                    'open': open_price, 'high': high_price,
                    'low': low_price, 'close': close_price,
                    'timestamp': datetime(2024, 1, 1) + timedelta(hours=i+j+1)
                })
                current_price = close_price
            
            # Exciting drop
            open_price = current_price
            close_price = current_price * 0.92
            high_price = open_price
            low_price = close_price
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i+3)
            })
            current_price = close_price
        else:
            # Normal candle with small movement
            open_price = current_price
            direction = np.random.choice([-1, 1])
            volatility = np.random.uniform(0.005, 0.02)
            close_price = current_price * (1 + direction * volatility)
            high_price = max(open_price, close_price) * (1 + np.random.uniform(0, 0.01))
            low_price = min(open_price, close_price) * (1 - np.random.uniform(0, 0.01))
            candles.append({
                'open': open_price, 'high': high_price,
                'low': low_price, 'close': close_price,
                'timestamp': datetime(2024, 1, 1) + timedelta(hours=i)
            })
            current_price = close_price
    
    return candles

# Generate data for each symbol
market_data = {}
for symbol in SYMBOLS:
    base_price = 50000 if 'BTC' in symbol else 3000
    market_data[symbol] = generate_synthetic_candles(symbol, num_candles=300, base_price=base_price)
    print(f"✓ Generated {len(market_data[symbol])} candles for {symbol}")

print(f"\nFirst candle for BTC/USDT:")
first_candle = market_data['BTC/USDT'][0]
print(f"  O: ${first_candle['open']:.2f}, H: ${first_candle['high']:.2f}, L: ${first_candle['low']:.2f}, C: ${first_candle['close']:.2f}")

✓ Generated 354 candles for BTC/USDT
✓ Generated 354 candles for ETH/USDT

First candle for BTC/USDT:
  O: $50000.00, H: $50091.72, L: $48769.35, C: $49152.59


## Initialize Strategy Parameters

In [25]:
# Create strategy parameters object
params = SupplyDemandParameters(**STRATEGY_PARAMS)

print("✓ Strategy parameters initialized")
print(f"  Min setup score: {params.min_setup_score}")
print(f"  Risk per trade: {params.risk_pct * 100}%")
print(f"  Min R:R ratio: {params.min_reward_risk}:1")

✓ Strategy parameters initialized
  Min setup score: 6.0
  Risk per trade: 2.0%
  Min R:R ratio: 3.0:1


## Detect Supply and Demand Zones

In [26]:
# Detect zones for each symbol
zones_by_symbol = {}

for symbol in SYMBOLS:
    candles = market_data[symbol]
    zones = detect_zones_dbr_rbd(candles, params)
    
    # Update freshness for all zones
    current_idx = len(candles) - 1
    for zone in zones:
        is_zone_fresh(zone, candles, current_idx)
    
    zones_by_symbol[symbol] = zones
    
    print(f"\n{symbol}:")
    print(f"  Total zones detected: {len(zones)}")
    demand_zones = [z for z in zones if z.zone_type == ZoneType.DEMAND]
    supply_zones = [z for z in zones if z.zone_type == ZoneType.SUPPLY]
    print(f"  Demand zones: {len(demand_zones)}")
    print(f"  Supply zones: {len(supply_zones)}")
    fresh_zones = [z for z in zones if z.is_fresh]
    print(f"  Fresh zones: {len(fresh_zones)}")

total_zones = sum(len(zones) for zones in zones_by_symbol.values())
print(f"\n✓ Total zones detected across all symbols: {total_zones}")


BTC/USDT:
  Total zones detected: 33
  Demand zones: 19
  Supply zones: 14
  Fresh zones: 4

ETH/USDT:
  Total zones detected: 40
  Demand zones: 18
  Supply zones: 22
  Fresh zones: 5

✓ Total zones detected across all symbols: 73


## Score Zones and Generate Trade Plans

In [27]:
# Score zones and build trade plans with multi-timeframe gating
trade_plans_by_symbol = {}
gating_stats = {
    'total_zones': 0,
    'passed_scoring': 0,
    'blocked_by_curve': 0,
    'blocked_by_trend': 0,
    'allowed_trades': 0,
    'curve_states': {'LOW': 0, 'EQ': 0, 'HIGH': 0},
    'trend_states': {'UP': 0, 'DOWN': 0, 'SIDEWAYS': 0}
}

for symbol in SYMBOLS:
    candles = market_data[symbol]
    zones = zones_by_symbol[symbol]
    current_price = candles[-1]['close']
    
    # Multi-timeframe analysis
    # For demo purposes, we'll use the same candles for all timeframes
    # In production, you would load actual HTF and ITF data
    
    # Detect curve state (HTF)
    supply_above, demand_below = find_nearest_fresh_zones_htf(candles, current_price, params)
    supply_proximal = supply_above.proximal if supply_above else None
    demand_proximal = demand_below.proximal if demand_below else None
    curve_state = classify_curve(current_price, supply_proximal, demand_proximal)
    gating_stats['curve_states'][curve_state] += 1
    
    # Detect trend state (ITF)
    pivot_highs, pivot_lows = detect_pivot_highs_lows(candles, params.pivot_len)
    trend_state = classify_trend(pivot_highs, pivot_lows, candles, params.pivots_to_consider)
    gating_stats['trend_states'][trend_state] += 1
    
    trade_plans = []
    
    for zone in zones:
        gating_stats['total_zones'] += 1
        
        # Update zone freshness
        is_zone_fresh(zone, candles, len(candles) - 1)
        
        # Calculate opposing zone for this zone
        if zone.zone_type == ZoneType.DEMAND:
            opposing_zone = supply_above
        else:
            opposing_zone = demand_below
        
        # Score the zone
        score = odds_enhancer_score(
            zone,
            current_price,
            curve_location(current_price, supply_above, demand_below),
            trend_direction_itf(candles, params),
            params,
            opposing_zone
        )
        
        # Check if score meets minimum threshold
        if score < params.min_setup_score:
            continue
        
        gating_stats['passed_scoring'] += 1
        
        # Apply multi-timeframe gating
        allowed, final_score = should_allow_trade(zone, curve_state, trend_state, score, params)
        
        if not allowed:
            # Track why trade was blocked
            if curve_state == "LOW" and zone.zone_type == ZoneType.SUPPLY:
                gating_stats['blocked_by_curve'] += 1
            elif curve_state == "HIGH" and zone.zone_type == ZoneType.DEMAND:
                gating_stats['blocked_by_curve'] += 1
            elif curve_state == "EQ":
                gating_stats['blocked_by_trend'] += 1
            continue
        
        gating_stats['allowed_trades'] += 1
        
        # Build trade plan
        trade_plan = build_trade_plan(
            zone,
            current_price,
            INITIAL_CAPITAL,
            params,
            opposing_zone,
            final_score
        )
        
        if trade_plan:
            trade_plans.append(trade_plan)
    
    trade_plans_by_symbol[symbol] = trade_plans

print("\n" + "="*60)
print("MULTI-TIMEFRAME GATING STATISTICS")
print("="*60)
print(f"Total zones detected: {gating_stats['total_zones']}")
print(f"Zones passing score threshold: {gating_stats['passed_scoring']}")
print(f"Trades blocked by curve gating: {gating_stats['blocked_by_curve']}")
print(f"Trades blocked by trend gating: {gating_stats['blocked_by_trend']}")
print(f"Trades allowed after gating: {gating_stats['allowed_trades']}")
print()
print("Curve State Distribution:")
for state, count in gating_stats['curve_states'].items():
    print(f"  {state}: {count}")
print()
print("Trend State Distribution:")
for state, count in gating_stats['trend_states'].items():
    print(f"  {state}: {count}")
print("="*60)

# Print trade plans summary
total_plans = sum(len(plans) for plans in trade_plans_by_symbol.values())
print(f"\n✓ Generated {total_plans} trade plans across {len(SYMBOLS)} symbols")
for symbol, plans in trade_plans_by_symbol.items():
    print(f"  {symbol}: {len(plans)} trade plans")



MULTI-TIMEFRAME GATING STATISTICS
Total zones detected: 73
Zones passing score threshold: 22
Trades blocked by curve gating: 5
Trades blocked by trend gating: 11
Trades allowed after gating: 6

Curve State Distribution:
  LOW: 1
  EQ: 1
  HIGH: 0

Trend State Distribution:
  UP: 0
  DOWN: 0
  SIDEWAYS: 2

✓ Generated 6 trade plans across 2 symbols
  BTC/USDT: 0 trade plans
  ETH/USDT: 6 trade plans


## Simulate Trades

In [28]:
# Simulate trade execution
def simulate_trades(symbol: str, candles: List[Dict], trade_plans: List, params: SupplyDemandParameters):
    """Simulate trade execution and calculate outcomes"""
    trades = []
    
    for tp in trade_plans:
        # For this simulation, assume price reaches entry
        # In reality, we'd need to check if price actually touched the zone
        entry_idx = tp.zone.created_at + 10  # Assume entry 10 candles after zone creation
        
        if entry_idx >= len(candles):
            continue
        
        # Decision made at least 1 candle after zone creation
        decision_idx = tp.zone.created_at + 1
        
        entry_time = candles[entry_idx]['timestamp']
        is_long = tp.zone.zone_type == ZoneType.DEMAND
        
        # Simulate outcome: randomly determine if stop or target hit first
        # In reality, we'd walk through candles to see which level is hit
        outcome_r = np.random.choice([
            -1.0,  # Stop hit (loss)
            params.take_profit_at_r  # Target hit (win)
        ], p=[0.35, 0.65])  # Assume 65% win rate for demo
        
        exit_price = tp.entry_price + (outcome_r * abs(tp.entry_price - tp.stop_loss) * (1 if is_long else -1))
        pnl = tp.position_size * (exit_price - tp.entry_price) * (1 if is_long else -1)
        
        # Calculate planned R (the R at decision time, before outcome)
        risk = abs(tp.entry_price - tp.stop_loss)
        reward = abs(tp.take_profit - tp.entry_price)
        planned_r = reward / risk if risk > 0 else 0
        
        trades.append({
            'symbol': symbol,
            'entry_time': entry_time,
            'exit_time': entry_time + timedelta(hours=20),  # Assume 20 hours
            'direction': 'LONG' if is_long else 'SHORT',
            'entry_price': tp.entry_price,
            'stop_loss': tp.stop_loss,
            'take_profit': tp.take_profit,
            'exit_price': exit_price,
            'position_size': tp.position_size,
            'outcome_r': outcome_r,  # Actual outcome R
            'r_multiple': planned_r,  # Use planned R for validation
            'planned_r': planned_r,  # R at decision time
            'pnl': pnl,
            'score': tp.score,
            'zone_type': tp.zone.zone_type.value,
            # Integrity check metadata
            'zone_created_at_idx': tp.zone.created_at,
            'zone_created_time': candles[tp.zone.created_at]['timestamp'] if tp.zone.created_at < len(candles) else None,
            'decision_idx': decision_idx,  # Decision made after zone creation
            'decision_time': candles[decision_idx]['timestamp'] if decision_idx < len(candles) else None,
            'entry_idx': entry_idx,
        })
    
    return trades

# Run simulation for all symbols
all_trades = []
for symbol in SYMBOLS:
    candles = market_data[symbol]
    trade_plans = trade_plans_by_symbol[symbol]
    trades = simulate_trades(symbol, candles, trade_plans, params)
    all_trades.extend(trades)
    print(f"{symbol}: {len(trades)} trades executed")

print(f"\n✓ Total trades executed: {len(all_trades)}")


BTC/USDT: 0 trades executed
ETH/USDT: 5 trades executed

✓ Total trades executed: 5


## Calculate Performance Metrics

## Backtest Integrity Report

Run integrity validation to ensure backtest quality.

In [29]:
# Run integrity checks
if len(all_trades) > 0:
    integrity_report = run_integrity_checks(
        trades=all_trades,
        min_r=params.min_reward_risk,
        r_tolerance=0.01
    )
    
    # Print the report
    print_integrity_report(integrity_report, verbose=True)
else:
    print("\n⚠ No trades to validate")


BACKTEST INTEGRITY REPORT

Total trades analyzed: 5
Status: ✗ VIOLATIONS FOUND

--------------------------------------------------------------------------------
Violation Summary:
--------------------------------------------------------------------------------
✗ Look Ahead: 3
✓ Entry Before Zone: 0
✓ R Calculation Mismatch: 0
✓ Insufficient R: 0

--------------------------------------------------------------------------------
Violation Details:
--------------------------------------------------------------------------------

[1] LOOK_AHEAD
    Reason: Trading decision made at 2024-01-04 19:00:00 before or at zone creation time 2024-01-04 21:00:00
    Symbol: ETH/USDT
    Entry Time: 2024-01-05 04:00:00
    Entry: $2879.80
    Zone Type: demand
    Details:
      - zone_created_at_idx: 105
      - decision_idx: 106
      - zone_created_time: 2024-01-04 21:00:00
      - decision_time: 2024-01-04 19:00:00

[2] LOOK_AHEAD
    Reason: Trading decision made at 2024-01-11 01:00:00 before or 

In [30]:
if len(all_trades) == 0:
    print("⚠ No trades executed. Try adjusting parameters or date range.")
else:
    # Convert to DataFrame for easier analysis
    df_trades = pd.DataFrame(all_trades)
    
    # Calculate metrics
    total_trades = len(df_trades)
    winning_trades = len(df_trades[df_trades['outcome_r'] > 0])
    losing_trades = len(df_trades[df_trades['outcome_r'] < 0])
    win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0
    
    avg_r = df_trades['outcome_r'].mean()
    avg_win_r = df_trades[df_trades['outcome_r'] > 0]['r_multiple'].mean() if winning_trades > 0 else 0
    avg_loss_r = df_trades[df_trades['outcome_r'] < 0]['r_multiple'].mean() if losing_trades > 0 else 0
    
    total_pnl = df_trades['pnl'].sum()
    
    # Calculate running equity and drawdown
    df_trades['cumulative_pnl'] = df_trades['pnl'].cumsum()
    df_trades['equity'] = INITIAL_CAPITAL + df_trades['cumulative_pnl']
    df_trades['peak_equity'] = df_trades['equity'].cummax()
    df_trades['drawdown'] = (df_trades['equity'] - df_trades['peak_equity']) / df_trades['peak_equity'] * 100
    max_drawdown = df_trades['drawdown'].min()
    
    final_equity = df_trades['equity'].iloc[-1]
    total_return = ((final_equity - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100
    
    # Print summary
    print("="*60)
    print("BACKTEST RESULTS SUMMARY")
    print("="*60)
    print(f"\nAccount Performance:")
    print(f"  Initial Capital:    ${INITIAL_CAPITAL:,.2f}")
    print(f"  Final Equity:       ${final_equity:,.2f}")
    print(f"  Total Return:       {total_return:.2f}%")
    print(f"  Total P&L:          ${total_pnl:,.2f}")
    print(f"  Max Drawdown:       {max_drawdown:.2f}%")
    
    print(f"\nTrade Statistics:")
    print(f"  Total Trades:       {total_trades}")
    print(f"  Winning Trades:     {winning_trades} ({win_rate:.1f}%)")
    print(f"  Losing Trades:      {losing_trades} ({100-win_rate:.1f}%)")
    
    print(f"\nR-Multiple Performance:")
    print(f"  Average R:          {avg_r:.2f}R")
    print(f"  Average Win:        {avg_win_r:.2f}R")
    print(f"  Average Loss:       {avg_loss_r:.2f}R")
    
    print(f"\nBreakdown by Symbol:")
    for symbol in SYMBOLS:
        symbol_trades = df_trades[df_trades['symbol'] == symbol]
        if len(symbol_trades) > 0:
            symbol_wins = len(symbol_trades[symbol_trades['r_multiple'] > 0])
            symbol_win_rate = (symbol_wins / len(symbol_trades) * 100)
            symbol_pnl = symbol_trades['pnl'].sum()
            print(f"  {symbol:12} {len(symbol_trades):3} trades | {symbol_win_rate:5.1f}% WR | ${symbol_pnl:+,.2f} P&L")
    
    print("="*60)

BACKTEST RESULTS SUMMARY

Account Performance:
  Initial Capital:    $10,000.00
  Final Equity:       $11,400.00
  Total Return:       14.00%
  Total P&L:          $1,400.00
  Max Drawdown:       -3.57%

Trade Statistics:
  Total Trades:       5
  Winning Trades:     3 (60.0%)
  Losing Trades:      2 (40.0%)

R-Multiple Performance:
  Average R:          1.40R
  Average Win:        10.56R
  Average Loss:       11.17R

Breakdown by Symbol:
  ETH/USDT       5 trades | 100.0% WR | $+1,400.00 P&L


## Example Trades with Details

In [31]:
if len(all_trades) > 0:
    print("\n" + "="*80)
    print("EXAMPLE TRADES (First 10)")
    print("="*80)
    
    for i, trade in enumerate(all_trades[:10], 1):
        print(f"\nTrade #{i}: {trade['symbol']} - {trade['direction']}")
        print(f"  Zone Type:       {trade['zone_type'].upper()}")
        print(f"  Setup Score:     {trade['score']:.2f}")
        print(f"  Entry Time:      {trade['entry_time']}")
        print(f"  Exit Time:       {trade['exit_time']}")
        print(f"  Entry Price:     ${trade['entry_price']:,.2f}")
        print(f"  Stop Loss:       ${trade['stop_loss']:,.2f}")
        print(f"  Take Profit:     ${trade['take_profit']:,.2f}")
        print(f"  Exit Price:      ${trade['exit_price']:,.2f}")
        print(f"  Position Size:   {trade['position_size']:.4f} units")
        print(f"  R Multiple:      {trade['r_multiple']:+.2f}R")
        print(f"  P&L:             ${trade['pnl']:+,.2f}")
        print(f"  Outcome:         {'WIN ✓' if trade['r_multiple'] > 0 else 'LOSS ✗'}")
    
    print("\n" + "="*80)
    print(f"Showing 10 of {len(all_trades)} total trades")
    print("="*80)
else:
    print("No trades to display")


EXAMPLE TRADES (First 10)

Trade #1: ETH/USDT - LONG
  Zone Type:       DEMAND
  Setup Score:     6.00
  Entry Time:      2024-01-05 04:00:00
  Exit Time:       2024-01-06 00:00:00
  Entry Price:     $2,879.80
  Stop Loss:       $2,840.84
  Take Profit:     $3,061.02
  Exit Price:      $2,996.67
  Position Size:   5.1340 units
  R Multiple:      +4.65R
  P&L:             $+600.00
  Outcome:         WIN ✓

Trade #2: ETH/USDT - LONG
  Zone Type:       DEMAND
  Setup Score:     6.00
  Entry Time:      2024-01-06 02:00:00
  Exit Time:       2024-01-06 22:00:00
  Entry Price:     $2,850.93
  Stop Loss:       $2,788.69
  Take Profit:     $3,061.02
  Exit Price:      $3,037.63
  Position Size:   3.2137 units
  R Multiple:      +3.38R
  P&L:             $+600.00
  Outcome:         WIN ✓

Trade #3: ETH/USDT - LONG
  Zone Type:       DEMAND
  Setup Score:     6.00
  Entry Time:      2024-01-08 23:00:00
  Exit Time:       2024-01-09 19:00:00
  Entry Price:     $2,619.46
  Stop Loss:       $2,578

## Trade Distribution Analysis

In [32]:
if len(all_trades) > 0:
    print("\nR-Multiple Distribution:")
    print("="*60)
    
    r_multiples = [trade['outcome_r'] for trade in all_trades]
    unique_rs = sorted(set(r_multiples))
    
    for r in unique_rs:
        count = r_multiples.count(r)
        pct = (count / len(all_trades)) * 100
        bar = '█' * int(pct / 2)
        print(f"  {r:+5.1f}R: {bar:30} {count:3} trades ({pct:5.1f}%)")
    
    print("\nTrade Direction Distribution:")
    print("="*60)
    longs = len([t for t in all_trades if t['direction'] == 'LONG'])
    shorts = len([t for t in all_trades if t['direction'] == 'SHORT'])
    print(f"  LONG:  {longs:3} trades ({longs/len(all_trades)*100:5.1f}%)")
    print(f"  SHORT: {shorts:3} trades ({shorts/len(all_trades)*100:5.1f}%)")
    
    print("\nZone Type Distribution:")
    print("="*60)
    demand_trades = len([t for t in all_trades if t['zone_type'] == 'demand'])
    supply_trades = len([t for t in all_trades if t['zone_type'] == 'supply'])
    print(f"  DEMAND: {demand_trades:3} trades ({demand_trades/len(all_trades)*100:5.1f}%)")
    print(f"  SUPPLY: {supply_trades:3} trades ({supply_trades/len(all_trades)*100:5.1f}%)")
else:
    print("No trade distribution to display")


R-Multiple Distribution:
   -1.0R: ████████████████████             2 trades ( 40.0%)
   +3.0R: ██████████████████████████████   3 trades ( 60.0%)

Trade Direction Distribution:
  LONG:    5 trades (100.0%)
  SHORT:   0 trades (  0.0%)

Zone Type Distribution:
  DEMAND:   5 trades (100.0%)
  SUPPLY:   0 trades (  0.0%)


## Summary and Recommendations

In [33]:
print("\n" + "="*80)
print("BACKTEST COMPLETE")
print("="*80)

if len(all_trades) > 0:
    print(f"\n✓ Successfully backtested Supply & Demand V1 strategy")
    print(f"✓ Analyzed {total_zones} zones across {len(SYMBOLS)} symbols")
    print(f"✓ Executed {len(all_trades)} trades with {win_rate:.1f}% win rate")
    print(f"✓ Average R multiple: {avg_r:.2f}R")
    print(f"✓ Max drawdown: {max_drawdown:.2f}%")
    
    print("\nKey Insights:")
    if win_rate >= 60:
        print("  • Win rate is strong (>60%) - strategy shows good edge")
    elif win_rate >= 50:
        print("  • Win rate is acceptable (50-60%) - monitor risk management")
    else:
        print("  • Win rate needs improvement (<50%) - consider adjusting parameters")
    
    if avg_r >= 1.0:
        print("  • Positive average R - strategy is profitable")
    else:
        print("  • Negative average R - review entry/exit criteria")
    
    if max_drawdown > -20:
        print("  • Drawdown is manageable - good risk control")
    else:
        print("  • Large drawdown detected - consider reducing position sizes")
    
    print("\nNext Steps:")
    print("  1. Test with real market data from exchange APIs")
    print("  2. Experiment with different parameter combinations")
    print("  3. Add multi-timeframe analysis (HTF curve, ITF trend)")
    print("  4. Implement proper entry fill logic (limit orders)")
    print("  5. Add trade management (breakeven moves, trailing stops)")
else:
    print("\n⚠ No trades were generated.")
    print("\nPossible reasons:")
    print("  • No zones met the minimum score threshold")
    print("  • Date range too narrow")
    print("  • Parameters too restrictive")
    print("\nTry:")
    print("  • Lowering min_setup_score in parameters")
    print("  • Extending the date range")
    print("  • Adjusting zone detection thresholds")

print("\n" + "="*80)


BACKTEST COMPLETE

✓ Successfully backtested Supply & Demand V1 strategy
✓ Analyzed 73 zones across 2 symbols
✓ Executed 5 trades with 60.0% win rate
✓ Average R multiple: 1.40R
✓ Max drawdown: -3.57%

Key Insights:
  • Win rate is strong (>60%) - strategy shows good edge
  • Positive average R - strategy is profitable
  • Drawdown is manageable - good risk control

Next Steps:
  1. Test with real market data from exchange APIs
  2. Experiment with different parameter combinations
  3. Add multi-timeframe analysis (HTF curve, ITF trend)
  4. Implement proper entry fill logic (limit orders)
  5. Add trade management (breakeven moves, trailing stops)

